In [1]:
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from pandas.api.types import is_numeric_dtype
import numpy as np
import pandas as pd
import re
from scipy import stats
from sklearn.preprocessing import Binarizer
from pandas.api.types import is_numeric_dtype
import datetime
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import math
from pandas import option_context
import numpy as np
from itertools import combinations
# Clustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from typing import List
import copy

from matplotlib.cm import get_cmap
from matplotlib.colors import rgb2hex
import matplotlib

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import ks_2samp
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.families.links import logit
from statsmodels.genmod.generalized_linear_model import GLM
from statsmodels.regression.linear_model import RegressionResultsWrapper
from statsmodels.genmod.generalized_linear_model import GLMResultsWrapper
from sklearn.metrics import roc_curve, auc

from pandas.tseries.offsets import DateOffset
import pandas as pd
import numpy as np
# from transformers_all import *
# from data_visualization import *

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.preprocessing import MinMaxScaler,StandardScaler

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2,mutual_info_classif

import imblearn
from collections import Counter
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
# from sklearn.metrics import plot_roc_curve
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import precision_recall_curve

from sklearn.metrics import RocCurveDisplay
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
# from catboost import CatBoostClassifier
from sklearn import metrics
from sklearn.feature_selection import f_classif
from sklearn.ensemble import VotingClassifier


from sklearn.metrics import roc_curve
from sklearn.metrics import recall_score, confusion_matrix, precision_score, f1_score, accuracy_score, classification_report

import simple_report as sr

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")


pd.set_option('display.max_columns', None)


In [14]:
df = pd.read_csv(r'C:\Users\douglas.sgrott_indic\Documents\Study\data-science-studies\1_dataset\telco\telco_churn_data.csv')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')


def augment_dataset_location_random(df, location_list, weights=None):
    _df = df.copy()
    if isinstance(weights, type(None)):
        _df['Location'] = np.random.choice(location_list, size=len(_df), p=weights)
        return _df
    _df['Location'] = np.random.choice(location_list, size=len(_df))
    return _df


def augment_dataset_location_cluster(df, features, n_clusters):
    # Cluster customers and assign each cluster a region:
    _df = df.copy()
    # X = _df[features].fillna(0)
    X = _df[features].fillna(_df[features].mean())
    X_scaled = StandardScaler().fit_transform(X)

    # Cluster into 6 groups
    kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(X_scaled)
    _df['Cluster'] = kmeans.labels_

    # Map clusters to locations
    cluster_to_location = {
        0: 'Chicago',
        1: 'New York',
        2: 'Boston',
        3: 'Miami',
        4: 'Los Angeles',
        5: 'Stanford',
    }
    _df['Location'] = _df['Cluster'].map(cluster_to_location)
    _df.drop(columns='Cluster', inplace=True)
    return _df


def augment_dataset_location_probabilistic(df, charge_col, config):
    location = np.empty(len(df), dtype=object)

    for level in config.values():
        lower, upper = level['range']
        mask = df[charge_col].between(lower, upper, inclusive='left')
        location[mask] = np.random.choice(
            level['choices'],
            size=mask.sum(),
            p=level['probs']
        )

    return location


def augment_dataset_datetime_random(df, start_date, end_date):
    _df = df.copy()
    n_rows = len(_df)
    random_days = np.random.randint(0, (end_date - start_date).days + 1, size=n_rows)
    _df['Date'] = start_date + pd.to_timedelta(random_days, unit='D')
    return _df


# Location Augmentation
# weights = [0.2, 0.25, 0.1, 0.15, 0.2, 0.1]
# location_list = ['Chicago', 'New York', 'Boston', 'Miami', 'Los Angeles', 'Stanford']

# features_to_cluster = ['MonthlyCharges', 'tenure', 'TotalCharges']
# n_clusters = 6

location_config = {
    'high': {
        'range': (80, np.inf),  # MonthlyCharges > 80
        'choices': ['New York', 'Los Angeles', 'Miami', 'Chicago', 'Boston', 'Stanford'],
        'probs':   [0.35, 0.35, 0.1, 0.1, 0.05, 0.05]
    },
    'mid': {
        'range': (60, 80),  # 60 < MonthlyCharges <= 80
        'choices': ['Chicago', 'Miami', 'New York', 'Los Angeles', 'Boston', 'Stanford'],
        'probs':   [0.3, 0.3, 0.15, 0.15, 0.05, 0.05]
    },
    'low': {
        'range': (0, 60),  # MonthlyCharges <= 60
        'choices': ['Boston', 'Stanford', 'Chicago', 'Miami', 'New York', 'Los Angeles'],
        'probs':   [0.4, 0.3, 0.1, 0.1, 0.05, 0.05]
    }
}


# Datetime Augmentation
start_date = pd.to_datetime('2023-01-01')
end_date = pd.to_datetime('2023-05-01') # 04-31 (4 months)



def expand_customer_history(df: pd.DataFrame) -> pd.DataFrame:
    """
    Expands a customer snapshot DataFrame into a historical, month-by-month view.

    For each customer, it generates a record for every month of their tenure,
    back-dating the 'Date' and recalculating 'TotalCharges' accordingly.

    Args:
        df (pd.DataFrame): The input DataFrame containing one row per customer
                           with their final tenure and date.

    Returns:
        pd.DataFrame: An expanded DataFrame with a full monthly history for each customer.
    """
    # Ensure the 'Date' column is a proper datetime object
    df['Date'] = pd.to_datetime(df['Date'])
    
    # List to hold all the new, historical records
    all_records = []

    # Iterate over each customer's final record
    for _, row in df.iterrows():
        final_tenure = row['tenure']
        final_date = row['Date']
        monthly_charges = row['MonthlyCharges']
        
        # Create a record for each month of the customer's tenure
        for i in range(1, final_tenure + 1):
            # Create a copy of the original row to modify
            new_record = row.copy()
            
            # 1. Set the tenure for this historical record
            new_record['tenure'] = i
            
            # 2. Calculate the date for this historical record
            # The date is calculated by offsetting backwards from the final date.
            # For a tenure of 1, the offset is (final_tenure - 1) months.
            months_to_offset = final_tenure - i
            new_record['Date'] = final_date - DateOffset(months=months_to_offset)
            
            # 3. Recalculate TotalCharges for this point in time
            # TotalCharges is simply the monthly charge multiplied by the current tenure.
            new_record['TotalCharges'] = round(monthly_charges * i, 2)
            
            # 4. Adjust the Churn status
            # A customer could only have churned in their final month.
            # All historical records before that must be 'No'.
            if i < final_tenure:
                new_record['Churn'] = 'No'
            
            # Add the newly created historical record to our list
            all_records.append(new_record)

    # Concatenate all records into a final DataFrame and sort for clarity
    expanded_df = pd.DataFrame(all_records)
    expanded_df = expanded_df.sort_values(by=['customerID', 'tenure']).reset_index(drop=True)
    
    return expanded_df


# df = augment_dataset_location_random(df, location_list, weights=weights)
# df = augment_dataset_location_cluster(df, features_to_cluster, n_clusters)
# df['Location'] = augment_dataset_location_probabilistic(df, 'MonthlyCharges', location_config)
df = augment_dataset_datetime_random(df, start_date, end_date)
df = df[df['tenure'] < 8].sample(1000)

df = expand_customer_history(df) [['customerID', 'Date', 'Churn', 'tenure',]]
# df.to_dict()

In [15]:

def create_cohort_table(df):
    date_col = 'Date'
    id_col = 'customerID'
    # Ensure your 'Date' column is datetime
    df[date_col] = pd.to_datetime(df[date_col])
    df['billing_month'] = df[date_col].dt.to_period('M')

    # Step 1: Get each customer's cohort month (first appearance)
    df['cohort_month'] = df.groupby(id_col)['billing_month'].transform('min')

    # Step 2: Drop duplicates (just keep one record per customer per month)
    df = df.drop_duplicates(subset=[id_col, 'billing_month'])

    # Step 3: Compute months since cohort start
    df['tenure_month'] = (df['billing_month'] - df['cohort_month']).apply(lambda x: x.n)

    # Step 4: Create pivoted cohort table
    cohort_table = (
        df.groupby(['cohort_month', 'tenure_month'])[id_col]
        .nunique()
        .unstack(level=1)  # FIX: Unstack tenure_month to become columns
        .sort_index(axis=0)  # FIX: Sort the index (cohorts)
    )

    # Fill NaNs with 0 for cleaner display
    # cohort_table = cohort_table.fillna(0).astype(int)
    return cohort_table

cohort_table = create_cohort_table(df)
cohort_table


tenure_month,0,1,2,3,4,5,6
cohort_month,,,,,,,
2022-07,15.0,15.0,15.0,15.0,15.0,15.0,15.0
2022-08,37.0,37.0,37.0,37.0,37.0,37.0,17.0
2022-09,61.0,61.0,61.0,61.0,61.0,35.0,20.0
2022-10,86.0,86.0,86.0,86.0,59.0,44.0,21.0
2022-11,104.0,104.0,104.0,72.0,42.0,17.0,NaN
2022-12,116.0,116.0,77.0,48.0,18.0,NaN,NaN
2023-01,173.0,78.0,49.0,16.0,1.0,NaN,NaN
2023-02,146.0,79.0,38.0,2.0,NaN,NaN,NaN
2023-03,150.0,42.0,2.0,NaN,NaN,NaN,NaN


In [16]:
import styler_extensions
from matplotlib.colors import LinearSegmentedColormap

max_ = cohort_table.max().max()
min_ = cohort_table.min().min()
custom_rwg = LinearSegmentedColormap.from_list('RedWhiteGreen', ['salmon', 'white', 'limegreen'])

cohort_table.style_ext.numeric_ranges(
    columns=cohort_table.columns,
    value_range=(min_, max_),
    cmap='coolwarm'
)


tenure_month,0,1,2,3,4,5,6
cohort_month,,,,,,,
2022-07,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
2022-08,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,17.000000
2022-09,61.000000,61.000000,61.000000,61.000000,61.000000,35.000000,20.000000
2022-10,86.000000,86.000000,86.000000,86.000000,59.000000,44.000000,21.000000
2022-11,104.000000,104.000000,104.000000,72.000000,42.000000,17.000000,nan
2022-12,116.000000,116.000000,77.000000,48.000000,18.000000,nan,nan
2023-01,173.000000,78.000000,49.000000,16.000000,1.000000,nan,nan
2023-02,146.000000,79.000000,38.000000,2.000000,nan,nan,nan
2023-03,150.000000,42.000000,2.000000,nan,nan,nan,nan
